In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"], "figure.dpi": 110})

SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
RES_DIR = "results/allen_wmb"
NAMES = {'ACA': 'Anterior cingulate area', 'AI': 'Agranular insular area', 'AUD': 'Auditory areas', 'AUD-TEa-PERI-ECT': 'Auditory/temporal/perirhinal/ectorhinal areas', 'CB': 'Cerebellum', 'CTXsp': 'Cortical subplate', 'ENT': 'Entorhinal area', 'HIP': 'Hippocampal region', 'HY': 'Hypothalamus', 'LSX': 'Lateral septal complex', 'MB': 'Midbrain', 'MY': 'Medulla', 'OLF': 'Olfactory areas', 'P': 'Pons', 'PAL': 'Pallidum', 'RHP': 'Retrohippocampal region', 'RSP': 'Retrosplenial area', 'SSp': 'Primary somatosensory area', 'STRd': 'Striatum dorsal region', 'STRv': 'Striatum ventral region', 'TH': 'Thalamus', 'VIS': 'Visual areas', 'sAMY': 'Striatum-like amygdalar nuclei', 'MOp': 'Primary motor area', 'VIS-PTLp': 'Visual/posterior parietal  areas', 'TEa-PERI-ECT': 'Temporal/perirhinal/ectorhinal areas ', 'PL-ILA-ORB': 'Prelimbic/infralimbic/orbital areas ', 'SS-GU-VISC': 'Somatosensory/gustatory/visceral areas', 'MO-FRP': 'Somatomotor - Frontal pole'}
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

_t20 = plt.get_cmap("tab20").colors
_all = [_t20[i] for i in range(0, 20, 2)] + [_t20[i] for i in range(1, 20, 2)] + list(plt.get_cmap("tab20b").colors)
PAL = [c for c in _all if not (abs(c[0] - c[1]) < 0.03 and abs(c[1] - c[2]) < 0.03)]

def class_num(c):
    n = pd.to_numeric(str(c)[:2], errors="coerce")
    return int(n) if pd.notna(n) else 0

def class_color(c):
    n = class_num(c)
    return PAL[(n - 1) % len(PAL)] if n > 0 else (0.5, 0.5, 0.5)

def is_neuron_class(c):
    return not (30 <= class_num(c) <= 34)

def neuron_type(c):
    c = str(c)
    if not is_neuron_class(c):
        return "non-neuronal"
    if "GABA" in c:
        return "GABAergic"
    if "Glut" in c:
        return "glutamatergic"
    return "other neuronal"

FOCUS_ROIS = ["PL-ILA-ORB", "ACA", "HIP", "CTXsp", "sAMY", "STRd", "STRv"]
PAIRS = [("Grm7", "Grm8"), ("Grm7", "Grm2"), ("Grm7", "Grm3"), ("Grm2", "Grm4")]
GROUP_GENES = {"Grm": ["Grm1", "Grm2", "Grm3", "Grm4", "Grm5", "Grm6", "Grm7", "Grm8"],
               "housekeeping": ["Gapdh", "Actb", "Eef1a1", "Tubb3", "Tuba1a", "Rpl13a", "B2m"],
               "other GPCRs": ["Gabbr1", "Gabbr2", "Gpr158", "Gprc5b", "Drd1", "Drd2", "Htr2a", "Chrm1", "Adora1", "Cnr1"],
               "long neuronal": ["Csmd1", "Nrxn1", "Nrxn3", "Cntnap2", "Ctnna2", "Ptprd", "Dlg2", "Auts2"]}

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

import pyarrow.parquet as pq

grp_of = {g: k for k, v in GROUP_GENES.items() for g in v}
order_genes = [g for v in GROUP_GENES.values() for g in v]
rows = []
for roi in FOCUS_ROIS:
    f = f"{SLIM_DIR}/{roi}_slim.parquet"
    if not os.path.exists(f):
        continue
    names = pq.ParquetFile(f).schema.names
    genes = [g for g in order_genes if g in names]
    d = pd.read_parquet(f, columns=["class"] + genes)
    d = d[[is_neuron_class(c) for c in d["class"]]]
    cpm = (np.exp2(d[genes].astype("float64")) - 1).mean()
    for g in genes:
        rows.append((roi, g, grp_of[g], float(cpm[g]), len(d)))
res = pd.DataFrame(rows, columns=["roi", "gene", "group", "mean_cpm", "n_neurons"])
res.to_csv(f"{RES_DIR}/scale_anchor_focus_regions.csv", index=False)
genes = [g for g in order_genes if g in set(res["gene"])]
M = res.pivot(index="roi", columns="gene", values="mean_cpm").reindex(index=[r for r in FOCUS_ROIS if r in set(res["roi"])], columns=genes)
nn = res.groupby("roi")["n_neurons"].first()
fig, ax = plt.subplots(figsize=(max(14, 0.62 * len(genes)), 0.95 * len(M) + 3.6))
im = ax.imshow(np.log10(M.to_numpy() + 1), aspect="auto", cmap="plasma")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M.iloc[i, j]
        ax.text(j, i, f"{v:.0f}" if v >= 10 else f"{v:.1f}", ha="center", va="center", fontsize=8, color="white" if im.cmap(im.norm(np.log10(v + 1)))[:3] and sum(im.cmap(im.norm(np.log10(v + 1)))[:3]) < 1.9 else "black")
ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=90, fontsize=10)
ax.xaxis.tick_top()
ax.set_yticks(range(len(M)))
ax.set_yticklabels([f"{NAMES.get(r, r)} ({r}); {nn[r]:,} neurons" for r in M.index], fontsize=10)
tr = ax.get_xaxis_transform()
for grp in GROUP_GENES:
    idx = [i for i, g in enumerate(genes) if grp_of[g] == grp]
    if idx:
        ax.plot([min(idx) - 0.4, max(idx) + 0.4], [1.1, 1.1], color="k", lw=1, transform=tr, clip_on=False)
        ax.text(np.mean(idx), 1.125, grp, ha="center", va="bottom", fontsize=12, transform=tr)
cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.05, pad=0.04)
cb.set_label("log10(mean linear expression + 1) in neurons (numbers in cells = mean of 2^x - 1)", fontsize=10)
ax.set_title("Is Grm expression low? Mean CPM of Grm1-8 in neurons vs housekeeping genes, other neuronal GPCRs and long neuronal genes, in the focus regions (Allen WMB, 10x)", fontsize=11, pad=95)
fig.savefig(f"{FIG_DIR}/ALL_fig6_scale_anchor.png", dpi=170, bbox_inches="tight")
plt.show()